# FPL Points Prediction - Model Evaluation

#### Team
- Andy Pike
- Mubeen Yaqub
- Sarjit Kaswala
- Sumedha Na

## 1. Background

### Preparing Our Data

FPL data is a time series so a temporal split is neceesary to avoid data leakage.
- **Train**: All historical seasons (2020–25) + Gameweeks 1–5 of 2025-26
- **Test**: Gameweeks 6–25 of 2025-26 (the season in progress)

We chose GW5 as the training cutoff for two reasons so new players joining mid-season need at least 5 games of history before the 5-game moving average features  become meaningful. Cutting earlier means most new players would have NaN features. Also a larger test set (GW6–25 = 20 gameweeks) gives more robust evaluation than just a handful of gameweeks. In production, this system would download the latest gameweek data and only predict the next game. However, for the purposes of testing out our models we predict multiple games at a time.

##### Interesting Parts of the Code

We decided engineer lag and moving average features on a per player basis to help the model learn the patterns in our time series. Lag 1 features react quickly to form changes but are noisy. MA5 features are more stable but lag behind sudden improvements or injuries. Using both gives the model short-term and medium-term signals simultaneously. 

```python
df = df.sort_values(['element', 'season', 'round']
df[f'{col}_lag1'] = df.groupby('element')[col].shift(1)
```

This ensures that for any row at Gameweek N, the features only contain information from GW N-1 and earlier. Without shift(1), the rolling average for GW N would include GW N's own score, which would be data leakage. To do this accurately we had to sort the gameweek data chronolgically and group by player (the FPL player identifier is 'element').F ull code for lag and moving average can be found in src/feature_engineering.py.  


We also applied an exponential decay to weight recent player performance more heavily. To do this we had to convert seasons and gameweeks (38 per season) into a tieline.

```
timeline = season_index × 38 + gameweek
```

Here's how that looks in practice:
- GW5 of 2025-26 → 5 × 38 + 5 = 195
- GW38 of 2024-25 → 4 × 38 + 38 = 190
- GW1 of 2020-21 → 0 × 38 + 1 = 1

We then calculated gameweeks ago (195 is gameweek 5 of the current season, which is our cutoff point.)

```
gameweeks_ago = 195 − timeline
```

So GW38 of 2024-25 is 195 − 190 = 5 gameweeks ago, and GW1 of 2020-21 is 195 − 1 = 194 gameweeks ago. This is then used to calculate the weight accordin to the formula below. -0.15 was chosen arbirtarily. We will experiment with tuning this parameter. These weights are stored in the sample_weight column of data/train.csv and passed to each model at fit time.

```
weight = exp(-0.15 × gameweeks_ago)
```

Weights are always between 0 and 1:
- gameweeks_ago = 0 -> exp(0) = 1.0` (maximum weight — the most recent training data)
- gameweeks_ago = 5 -> exp(-0.75) ≈ 0.47` (roughly half weight)
- gameweeks_ago = 38 -> exp(-5.7) ≈ 0.003 (near zero - effectively ignord)

### Features Selected

**Target variable:** `total_points` — the FPL points a player scores in a given gameweek. This is the raw points total including goals, assists, clean sheets, bonus points, and deductions.

We chose the following features based on the correlation analysis in our EDA:

| Feature | Type | Why it was chosen |
|---|---|---|
| `total_points_lag1` | lag1 | Direct signal of recent output; captures the immediate momentum of a player's last performance |
| `total_points_ma5` | ma5 | Smoothed points trend; reduces noise from one-off hauls or blanks |
| `expected_goal_involvements_lag1` | lag1 | xGI (xG + xA) reflects underlying attacking threat more reliably than actual goals/assists, which are noisier |
| `expected_goal_involvements_ma5` | ma5 | Sustained attacking involvement over 5 weeks — a better signal of consistent returns than a single week |
| `minutes_lag1` | lag1 | Availability proxy; a player who played 90 minutes last week is more likely to start this week |
| `ict_index_ma5` | ma5 | FPL's composite influence/creativity/threat score; captures attacking involvement beyond xGI alone |
| `bps_ma5` | ma5 | Bonus point system score; reflects consistently high-quality performances that earn bonus points |

### Models Selected

We wanted to use Random Forest and Gradient Boosting because they are highly accurate at modeling complex non-linear relationships in tabular data with minimal preprocessing or hyperparameter tuning. We want to compliment these models with a linear model as a baseline. We just Ridge for this purpose. Additionally we used two naive models as additional baselines: lag1 (ie this week's points will be the same as last week) and MA5 (this week's points will be the average points from the last 5 weeks)

### Evaluation metrics

Our primary evaluation metric is Mean Absolute Error. This measures the average of |actual − predicted| across all player-gameweeks. Our goal is to predict the number of points each player will score. Ths metric is highly interpretable. An MAE of 1.03 means the model is off by about 1 point per player per gameweek on average. We also use RMSE which is the square root of the average of (actual − predicted)². This metric heavily penalizes large errors. This is important because in FPL we can only choose 11 players and we are trying to identify who will score the most points each weeks. We also use R², which measures how much of the variation in player scores the model explains. 

## 2.Results

In [32]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

test_df = pd.read_csv('../data/test.csv')
predictions_df = pd.read_csv('../data/predictions.csv')

with open('../metrics/results.json', 'r') as f:
    metrics = json.load(f)


# Inspect test set
print(f"Test set: {len(test_df)} rows")
print(f"Gameweeks: {test_df['round'].min()} to {test_df['round'].max()}")

Test set: 16187 rows
Gameweeks: 6 to 26


In [33]:
# Display metrics from JSON
metrics_df = pd.DataFrame(metrics).T
metrics_df = metrics_df.sort_values('mae')
print("Model Performance (sorted by MAE):")
print(metrics_df.to_string())

Model Performance (sorted by MAE):
                      mae    rmse      r2
random_forest      1.0307  1.9777  0.3077
gradient_boosting  1.0567  2.0037  0.2894
naive_ma5          1.0573  2.1159  0.2076
ridge_regression   1.0772  1.9733  0.3108
naive_lag1         1.1494  2.5968 -0.1935


## 3. Observations and Conclusions

